In [ ]:
## Generate summary results
'''
This notebook processes all realizations generated by the C++
simulation and extracts the quantities required for the ensemble
analysis.

The outputs are two files named

Results.txt

ResultsTau.txt

which are used by the next notebook
(analyze_ensemble.ipynb).
'''

In [1]:
###########################################################
# Imports
###########################################################

import numpy as np
from pathlib import Path

In [ ]:
###########################################################
# Parameters
###########################################################

DATA_DIR = Path("../sample_output/ensemble")

R_MAX = 2000  # maximum number of run realizations

TAU_MAX = 7

N = 2000

ETA = 0.02

N_E = int(0.8 * N)

D = 5

TE = 5

TI = 7

W_E = 1

W_I = -4

In [ ]:
###########################################################
# Initialize output table
###########################################################

results = []

In [ ]:
###########################################################
# Process all realizations
###########################################################

for R in range(1, R_MAX + 1):

    #######################################################
    # Load activity
    #######################################################

    activity = np.loadtxt(
        DATA_DIR / f"ActivityInTime_{R}.txt"
    )

    rho = activity[:, 1]

    tz = np.argmax(rho == 0)

    next_cycle = int(np.sum(rho[tz:]) > 0.2)

    #######################################################
    # Load active neurons at different τ
    #######################################################

    active_tau = []

    with open(DATA_DIR / f"ActiveNodesInTau_{R}.txt") as f:

        for line in f:

            row = np.fromstring(
                line,
                sep=" "
            )

            active_tau.append(row[1:])

    #######################################################
    # Count active excitatory / inhibitory neurons
    #######################################################

    N_E_active = np.zeros(
        TAU_MAX + 1,
        dtype=int
    )

    N_I_active = np.zeros(
        TAU_MAX + 1,
        dtype=int
    )

    for tau in range(TAU_MAX + 1):

        neurons = active_tau[tau]

        N_E_active[tau] = np.sum(neurons < N_E)

        N_I_active[tau] = np.sum(neurons >= N_E)

    #######################################################
    # Load ghost neurons
    #######################################################

    ghost_tau = []

    with open(DATA_DIR / f"GhostNodesInTau_{R}.txt") as f:

        for line in f:

            row = np.fromstring(
                line,
                sep=" "
            )

            ghost_tau.append(row[1:])

    #######################################################
    # Count fresh neurons
    #######################################################

    N_E_fresh = np.zeros(
        TAU_MAX + 1,
        dtype=int
    )

    N_I_fresh = np.zeros(
        TAU_MAX + 1,
        dtype=int
    )

    for tau in range(TAU_MAX + 1):

        active = active_tau[tau]

        ghost = ghost_tau[tau]

        fresh = np.setdiff1d(
            active,
            ghost
        )

        N_E_fresh[tau] = np.sum(fresh < N_E)

        N_I_fresh[tau] = np.sum(fresh >= N_E)

    #######################################################
    # Integrate over τ
    #######################################################

    N_E_tau = np.sum(N_E_active)

    N_I_tau = np.sum(N_I_active)

    N_E_fresh_tau = np.sum(N_E_fresh)

    N_I_fresh_tau = np.sum(N_I_fresh)

    #######################################################
    # Store one realization
    #######################################################

    results.append([
        R,
        next_cycle,
        N_E_tau,
        N_E_fresh_tau,
        N_I_tau,
        N_I_fresh_tau
    ])

In [ ]:
###########################################################
# Save summary table
###########################################################

output_file = DATA_DIR / "Results.txt"

header = (
    "#R\t"
    "NextCycle\t"
    "NE(tau)\t"
    "NEfresh(tau)\t"
    "NI(tau)\t"
    "NIfresh(tau)"
)

with open(output_file, "w") as f:

    f.write(header + "\n")

    for row in results:

        f.write(
            "\t".join(map(str, row))
        )

        f.write("\n")

print()
print(f"Results successfully saved to:")
print(output_file)

In [ ]:
###########################################################
# Preview results
###########################################################

results = np.loadtxt(
    DATA_DIR / "Results.txt",
    skiprows=1
)

print(results.shape)

results[:10]

In [2]:
###########################################################
# Imports
###########################################################

from pathlib import Path

In [4]:
###########################################################
# Parameters
###########################################################

DATA_DIR = Path("../sample_output")

OUTPUT_DIR = Path("../processed_data")

TAU_MAX = 7

N_E = 1600

In [5]:
results_tau = np.zeros(
    (TAU_MAX + 1, 5),
    dtype=int
)

results_tau[:,0] = np.arange(TAU_MAX + 1)

In [8]:
###########################################################
# Build ensemble tau statistics
###########################################################

activity_files = sorted(
    DATA_DIR.glob("ActivityInTime_*.txt")
)

print(f"{len(activity_files)} realizations found.")

for activity_file in activity_files:

    realization = activity_file.stem.split("_")[-1]

    #######################################################
    # Active neurons at each tau
    #######################################################

    active_nodes_tau = []

    with open(
        DATA_DIR / f"ActiveNodesInTau_{realization}.txt"
    ) as f:

        for line in f:

            row = np.fromstring(
                line,
                sep=" ",
                dtype=int
            )

            active_nodes_tau.append(row[1:])

    #######################################################
    # Ghost neurons at each tau
    #######################################################

    ghost_nodes_tau = []

    with open(
        DATA_DIR / f"GhostNodesInTau_{realization}.txt"
    ) as f:

        for line in f:

            row = np.fromstring(
                line,
                sep=" ",
                dtype=int
            )

            ghost_nodes_tau.append(row[1:])

    #######################################################
    # Accumulate statistics
    #######################################################

    for tau in range(TAU_MAX + 1):

        active = active_nodes_tau[tau]

        ghost = ghost_nodes_tau[tau]

        fresh = np.setdiff1d(
            active,
            ghost
        )

        active_E = np.sum(active < N_E)

        active_I = np.sum(active >= N_E)

        fresh_E = np.sum(fresh < N_E)

        fresh_I = np.sum(fresh >= N_E)

        results_tau[tau,1] += active_E

        results_tau[tau,2] += fresh_E

        results_tau[tau,3] += active_I

        results_tau[tau,4] += fresh_I

print("Finished building tau statistics.")

2 realizations found.
Finished building tau statistics.


In [9]:
def save_results_tau(filename, matrix):

    with open(filename, "w") as f:

        f.write(
            "#tau\t"
            "NE(tau)\t"
            "NEfresh(tau)\t"
            "NI(tau)\t"
            "NIfresh(tau)\n"
        )

        for row in matrix:

            f.write(
                "\t".join(map(str, row))
            )

            f.write("\n")

In [10]:
save_results_tau(
    OUTPUT_DIR / "ResultsTau.txt",
    results_tau
)

print("ResultsTau.txt saved successfully.")

ResultsTau.txt saved successfully.
